In [ ]:
"""Imports and plotting setup"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['ps.useafm'] = True
plt.rcParams['pdf.use14corefonts'] = True
from torch.autograd import grad
import scipy.io
from scipy.interpolate import griddata
from scipy.stats.qmc import LatinHypercube
import math
from collections import deque, defaultdict
import os
import time

In [ ]:
"""Runtime setup"""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(1234)
torch.manual_seed(1234)


In [ ]:
"""Calculation Cost"""
import json
import platform
import datetime
try:
    import psutil
except ImportError:
    psutil = None

COST_OUTPUT_DIR = "./open_source_assets"
COST_REPORT_PREFIX = "ns_darcy_architecture3"

def sync_cuda_if_available():

    if torch.cuda.is_available():
        torch.cuda.synchronize()

def cost_timer():

    sync_cuda_if_available()
    return time.perf_counter()

def reset_cuda_peak_memory():

    if torch.cuda.is_available():
        sync_cuda_if_available()
        torch.cuda.reset_peak_memory_stats()

def get_cuda_memory_summary():

    if not torch.cuda.is_available():
        return {
            "cuda_available": False,
            "current_allocated_mb": None,
            "current_reserved_mb": None,
            "peak_allocated_mb": None,
            "peak_reserved_mb": None,
        }
    sync_cuda_if_available()
    return {
        "cuda_available": True,
        "current_allocated_mb": torch.cuda.memory_allocated() / 1024**2,
        "current_reserved_mb": torch.cuda.memory_reserved() / 1024**2,
        "peak_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "peak_reserved_mb": torch.cuda.max_memory_reserved() / 1024**2,
    }

def get_cpu_memory_mb():

    if psutil is None:
        return None
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def get_hardware_info():

    info = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cpu_count_logical": os.cpu_count(),
        "cpu_rss_mb_at_report": get_cpu_memory_mb(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update({
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_count": torch.cuda.device_count(),
            "gpu_total_memory_mb": props.total_memory / 1024**2,
            "gpu_compute_capability": f"{props.major}.{props.minor}",
        })
    else:
        info.update({
            "gpu_name": None,
            "gpu_count": 0,
            "gpu_total_memory_mb": None,
            "gpu_compute_capability": None,
        })
    return info

def count_trainable_parameters(model):

    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def _json_default(obj):

    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return str(obj)

def write_computational_cost_report(report, prefix=COST_REPORT_PREFIX):

    os.makedirs(COST_OUTPUT_DIR, exist_ok=True)
    json_path = os.path.join(COST_OUTPUT_DIR, f"{prefix}_computational_cost.json")
    csv_path = os.path.join(COST_OUTPUT_DIR, f"{prefix}_computational_cost_summary.csv")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2, default=_json_default)

    training = report.get("training", {})
    inference = report.get("inference", {})
    rl = report.get("rl_overhead", {})
    model_info = report.get("model", {})
    memory = report.get("memory", {})
    hardware = report.get("hardware", {})

    summary = {
        "method": report.get("method", "Architecture III"),
        "gpu_name": hardware.get("gpu_name"),
        "trainable_parameters": model_info.get("trainable_parameters"),
        "stage1_time_s": training.get("stage1_time_s"),
        "stage2_time_s": training.get("stage2_time_s"),
        "training_time_s": training.get("total_training_time_s"),
        "training_gpu_hours": training.get("gpu_hours"),
        "peak_gpu_memory_allocated_mb": memory.get("training_peak_allocated_mb"),
        "peak_gpu_memory_reserved_mb": memory.get("training_peak_reserved_mb"),
        "rl_inference_time_s": rl.get("policy_inference_time_s"),
        "rl_update_time_s": rl.get("policy_update_time_s"),
        "rl_total_overhead_time_s": rl.get("total_measured_overhead_time_s"),
        "rl_overhead_fraction": rl.get("overhead_fraction_of_training_time"),
        "prediction_time_s": inference.get("prediction_time_s"),
        "prediction_points": inference.get("prediction_points"),
        "prediction_time_per_point_s": inference.get("prediction_time_per_point_s"),
    }
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        f.write(",".join(summary.keys()) + "\n")
        f.write(",".join("" if v is None else str(v) for v in summary.values()) + "\n")

    return {"json": json_path, "csv": csv_path}


In [ ]:
"""Reward functions"""


def calculate_reward(current_losses, previous_losses, initial_losses):
    if previous_losses is None:
        return 0.0

    total_current = sum(current_losses)
    total_previous = sum(previous_losses)

    if total_previous == 0:
        return 0.0

    improvement = (total_previous - total_current) / total_previous

    if improvement > 0.08:
        base_reward = 2.5 * improvement
    elif improvement > 0.04:
        base_reward = 2.2 * improvement
    elif improvement > 0.02:
        base_reward = 2.0 * improvement
    elif improvement > 0.01:
        base_reward = 1.8 * improvement
    elif improvement > -0.01:
        base_reward = 0.0
    else:
        base_reward = -2.0 * abs(improvement)

    if total_current < 0.05:
        absolute_reward = 1.0
    elif total_current < 0.1:
        absolute_reward = 0.8
    elif total_current < 0.5:
        absolute_reward = 0.5
    elif total_current < 1.0:
        absolute_reward = 0.2
    elif total_current > 20.0:
        absolute_reward = -0.3
    elif total_current > 50.0:
        absolute_reward = -0.5
    else:
        absolute_reward = 0.0

    balance_reward = 0.0
    if len(current_losses) > 1:
        max_loss = max(current_losses)
        min_loss = min(current_losses)
        if max_loss > 0:
            balance_ratio = min_loss / max_loss
            if balance_ratio > 0.1:
                balance_reward = 0.3
            elif balance_ratio < 0.01:
                balance_reward = -0.3

    stability_reward = 0.0
    if abs(improvement) < 0.005:
        if total_current < 1.0:
            stability_reward = 0.2

    total_reward = base_reward + absolute_reward + balance_reward + stability_reward

    return max(min(total_reward, 3.0), -3.0)

def calculate_arch_reward(pre_losses, post_losses, total_params, action_type=None,
                         current_depth=None, current_units=None):

    pre_total = sum(pre_losses)
    post_total = sum(post_losses)

    if pre_total == 0:
        return 0.0

    improvement = (pre_total - post_total) / pre_total

    if improvement > 0.10:
        base_reward = 2.5 * improvement
    elif improvement > 0.06:
        base_reward = 2.0 * improvement
    elif improvement > 0.03:
        base_reward = 1.5 * improvement
    elif improvement > 0.01:
        base_reward = 1.0 * improvement
    elif improvement > -0.02:
        base_reward = 0.0
    else:
        base_reward = -2.5 * abs(improvement)

    targeted_reward = 0.0

    if action_type and len(pre_losses) >= 7 and len(post_losses) >= 7:

        pre_ns_loss = pre_losses[1] + pre_losses[4] + pre_losses[6]
        post_ns_loss = post_losses[1] + post_losses[4] + post_losses[6]

        pre_darcy_loss = pre_losses[0] + pre_losses[3] + pre_losses[5]
        post_darcy_loss = post_losses[0] + post_losses[3] + post_losses[5]

        ns_improvement = 0.0
        darcy_improvement = 0.0

        if pre_ns_loss > 1e-10:
            ns_improvement = (pre_ns_loss - post_ns_loss) / pre_ns_loss
        if pre_darcy_loss > 1e-10:
            darcy_improvement = (pre_darcy_loss - post_darcy_loss) / pre_darcy_loss

        if pre_ns_loss > pre_darcy_loss * 1.5:

            if 'ns_' in action_type:
                if ns_improvement > 0.05:
                    targeted_reward = 1.2
                elif ns_improvement > 0.02:
                    targeted_reward = 0.8
                elif ns_improvement > 0:
                    targeted_reward = 0.4
                else:
                    targeted_reward = -0.6
            elif 'darcy_' in action_type:
                targeted_reward = -0.5

        elif pre_darcy_loss > pre_ns_loss * 1.5:

            if 'darcy_' in action_type:
                if darcy_improvement > 0.05:
                    targeted_reward = 1.2
                elif darcy_improvement > 0.02:
                    targeted_reward = 0.8
                elif darcy_improvement > 0:
                    targeted_reward = 0.4
                else:
                    targeted_reward = -0.6
            elif 'ns_' in action_type:
                targeted_reward = -0.5
        else:

            if 'ns_' in action_type and ns_improvement > 0:
                targeted_reward = 0.5
            elif 'darcy_' in action_type and darcy_improvement > 0:
                targeted_reward = 0.5

    interface_reward = 0.0

    if len(pre_losses) >= 3 and len(post_losses) >= 3:
        pre_interface_loss = pre_losses[2]
        post_interface_loss = post_losses[2]

        if pre_interface_loss > 1e-10:
            interface_improvement = (pre_interface_loss - post_interface_loss) / pre_interface_loss

            if interface_improvement > 0.15:
                interface_reward = 1.0
            elif interface_improvement > 0.08:
                interface_reward = 0.7
            elif interface_improvement > 0.03:
                interface_reward = 0.4
            elif interface_improvement > -0.05:
                interface_reward = 0.0
            else:
                interface_reward = -0.8

        if post_interface_loss < 0.05:
            interface_reward += 0.5
        elif post_interface_loss < 0.10:
            interface_reward += 0.3
        elif post_interface_loss > 0.50:
            interface_reward -= 0.4

    param_reward = 0.0
    if total_params:
        if total_params > 600000:
            param_reward = -0.6
        elif total_params > 450000:
            param_reward = -0.3
        elif total_params < 200000:
            param_reward = 0.3
        elif total_params < 100000:
            param_reward = 0.1

    total_reward = (
        base_reward * 0.35 +
        targeted_reward * 0.35 +
        interface_reward * 0.20 +
        param_reward * 0.10
    )

    return max(min(total_reward, 3.0), -3.0)

In [ ]:
"""Domain classifier"""

class DomainClassifier:

    def __init__(self, interface_tolerance=1e-6):
        self.interface_tolerance = interface_tolerance

    def classify_points(self, x, y):

        if isinstance(x, torch.Tensor):
            x = x.detach().cpu().numpy().flatten()
        if isinstance(y, torch.Tensor):
            y = y.detach().cpu().numpy().flatten()

        classifications = []
        for xi, yi in zip(x, y):
            if abs(yi) < self.interface_tolerance:
                classifications.append('interface')
            elif yi < -self.interface_tolerance:
                classifications.append('ns')
            else:
                classifications.append('darcy')
        return classifications

    def get_ns_mask(self, x, y):
        classifications = self.classify_points(x, y)
        return np.array([c == 'ns' for c in classifications])

    def get_darcy_mask(self, x, y):
        classifications = self.classify_points(x, y)
        return np.array([c == 'darcy' for c in classifications])

    def get_interface_mask(self, x, y):
        classifications = self.classify_points(x, y)
        return np.array([c == 'interface' for c in classifications])


In [ ]:
"""RL controller"""


class RLController:

    def __init__(self, n_actions=28, state_dim=19, hidden_dim=64,  min_units=5, max_units=256, min_depth=2, max_depth=10):

        self.policy_net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, n_actions),
            torch.nn.Softmax(dim=-1)
        ).to(device)

        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.gamma = 0.99

        self.min_units = min_units
        self.max_units = max_units
        self.min_depth = min_depth
        self.max_depth = max_depth

        self.min_base_points = 1000
        self.max_base_points = 10000
        self.max_refined_points = 10000

        self.episode_rewards = []
        self.episode_log_probs = []
        self.action_stats = {
            'weight': 0, 'ns_arch': 0, 'darcy_arch': 0, 'sampling': 0, 'total': 0
        }
        self.arch_action_stats = {

            'ns_increase_depth': 0, 'ns_increase_width': 0,
            'ns_decrease_depth': 0, 'ns_decrease_width': 0,

            'darcy_increase_depth': 0, 'darcy_increase_width': 0,
            'darcy_decrease_depth': 0, 'darcy_decrease_width': 0,
            'total_arch': 0, 'total_decisions': 0
        }

    def get_enhanced_state(self, losses, ns_depth, ns_units, darcy_depth, darcy_units,
                          loss_history, darcy_base_points, ns_base_points,
                          darcy_refined_points, ns_refined_points):

        state = []

        state.extend(losses)

        state.append(ns_depth / 10.0)
        state.append(ns_units / 256.0)

        state.append(darcy_depth / 10.0)
        state.append(darcy_units / 256.0)

        training_stability = 0.0
        if len(loss_history) > 5:
            recent_losses = loss_history[-5:]
            loss_std = np.std(recent_losses)
            if loss_std < 0.001:
                training_stability = 1.0
            elif loss_std > 0.1:
                training_stability = -1.0
        state.append(training_stability)

        improvement_trend = 0.0
        if len(loss_history) > 10:
            recent_10 = loss_history[-10:]
            improvement_trend = (recent_10[0] - recent_10[-1]) / (recent_10[0] + 1e-9)
        state.append(improvement_trend)

        medium_term_improvement = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            medium_term_improvement = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
        state.append(medium_term_improvement)

        stagnation_indicator = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            improvement_50 = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
            if abs(improvement_50) < 0.02:
                stagnation_indicator = 1.0
        state.append(stagnation_indicator)

        state.append(darcy_base_points / self.max_base_points)
        state.append(darcy_refined_points / self.max_refined_points)

        state.append(ns_base_points / self.max_base_points)
        state.append(ns_refined_points / self.max_refined_points)

        return torch.FloatTensor(state).to(device).unsqueeze(0)

    def get_valid_actions(self, ns_depth, ns_units, darcy_depth, darcy_units,
                         darcy_base_points, ns_base_points,
                         darcy_refined_points, ns_refined_points):
        valid = [1.0] * 28

        if ns_depth >= self.max_depth:
            valid[14] = 0.0
        if ns_depth <= self.min_depth:
            valid[15] = 0.0
        if ns_units >= self.max_units:
            valid[16] = 0.0
        if ns_units <= self.min_units:
            valid[17] = 0.0

        if darcy_depth >= self.max_depth:
            valid[18] = 0.0
        if darcy_depth <= self.min_depth:
            valid[19] = 0.0
        if darcy_units >= self.max_units:
            valid[20] = 0.0
        if darcy_units <= self.min_units:
            valid[21] = 0.0

        if darcy_refined_points >= self.max_refined_points:
            valid[22] = 0.0
        if darcy_base_points >= self.max_base_points:
            valid[23] = 0.0
        if darcy_base_points <= self.min_base_points:
            valid[24] = 0.0

        if ns_refined_points >= self.max_refined_points:
            valid[25] = 0.0
        if ns_base_points >= self.max_base_points:
            valid[26] = 0.0
        if ns_base_points <= self.min_base_points:
            valid[27] = 0.0

        return valid

    def select_action(self, state):

        if not isinstance(state, torch.Tensor):
            state_tensor = torch.FloatTensor(state).to(device)
        else:
            state_tensor = state

        probs = self.policy_net(state_tensor)

        state_values = state_tensor.squeeze(0)
        ns_depth = int(state_values[7].item() * 10)
        ns_units = int(state_values[8].item() * 256)
        darcy_depth = int(state_values[9].item() * 10)
        darcy_units = int(state_values[10].item() * 256)
        darcy_base = int(state_values[15].item() * self.max_base_points)
        darcy_refined = int(state_values[16].item() * self.max_refined_points)
        ns_base = int(state_values[17].item() * self.max_base_points)
        ns_refined = int(state_values[18].item() * self.max_refined_points)

        valid_actions = self.get_valid_actions(
            ns_depth, ns_units, darcy_depth, darcy_units,
            darcy_base, ns_base, darcy_refined, ns_refined
        )
        masked_probs = probs * torch.FloatTensor(valid_actions).to(device)
        masked_probs = masked_probs / (masked_probs.sum() + 1e-9)

        self.action_stats['total'] += 1

        action_dist = torch.distributions.Categorical(masked_probs)
        action = action_dist.sample()

        action_id = action.item()
        if action_id < 14:
            self.action_stats['weight'] += 1
        elif action_id < 18:
            self.action_stats['ns_arch'] += 1
            self.arch_action_stats['total_decisions'] += 1
            self.arch_action_stats['total_arch'] += 1
            action_names = ['ns_increase_depth', 'ns_decrease_depth',
                          'ns_increase_width', 'ns_decrease_width']
            action_name = action_names[action_id - 14]
            self.arch_action_stats[action_name] += 1
            print(f" NSarchitecture adjustment: {action_name}")
        elif action_id < 22:
            self.action_stats['darcy_arch'] += 1
            self.arch_action_stats['total_decisions'] += 1
            self.arch_action_stats['total_arch'] += 1
            action_names = ['darcy_increase_depth', 'darcy_decrease_depth',
                          'darcy_increase_width', 'darcy_decrease_width']
            action_name = action_names[action_id - 18]
            self.arch_action_stats[action_name] += 1
            print(f" Darcyarchitecture adjustment: {action_name}")
        else:
            self.action_stats['sampling'] += 1
            sampling_names = {
                22: "Darcyrefinedsampling", 23: "Darcybase+20%", 24: "Darcybase-20%",
                25: "NSrefinedsampling", 26: "NSbase+20%", 27: "NSbase-20%"
            }
            print(f" {sampling_names.get(action_id, 'unknown sampling')}")

        return action_id, action_dist.log_prob(action)

    def create_architecture_command(self, action_id):

        commands = {

            14: {'type': 'ns_depth', 'direction': 'increase'},
            15: {'type': 'ns_depth', 'direction': 'decrease'},
            16: {'type': 'ns_width', 'direction': 'increase'},
            17: {'type': 'ns_width', 'direction': 'decrease'},

            18: {'type': 'darcy_depth', 'direction': 'increase'},
            19: {'type': 'darcy_depth', 'direction': 'decrease'},
            20: {'type': 'darcy_width', 'direction': 'increase'},
            21: {'type': 'darcy_width', 'direction': 'decrease'}
        }
        return commands.get(action_id, None)

    def create_sampling_command(self, action_id):

        commands = {
            22: {'region': 'darcy', 'action': 'add_refined'},
            23: {'region': 'darcy', 'action': 'increase_base'},
            24: {'region': 'darcy', 'action': 'decrease_base'},
            25: {'region': 'ns', 'action': 'add_refined'},
            26: {'region': 'ns', 'action': 'increase_base'},
            27: {'region': 'ns', 'action': 'decrease_base'}
        }
        return commands.get(action_id, None)

    def update_policy(self):

        if len(self.episode_rewards) == 0:
            return

        discounted_rewards = []
        R = 0
        for r in reversed(self.episode_rewards):
            R = r + self.gamma * R
            discounted_rewards.insert(0, R)

        discounted_rewards = torch.tensor(discounted_rewards).to(device)
        if len(discounted_rewards) > 1:
            discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)

        policy_loss = []
        for log_prob, G in zip(self.episode_log_probs, discounted_rewards):
            policy_loss.append(-log_prob * G)

        self.optimizer.zero_grad()
        policy_loss = torch.stack(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()

        self.episode_rewards = []
        self.episode_log_probs = []

    def add_reward(self, reward):
        self.episode_rewards.append(reward)

    def add_log_prob(self, log_prob):
        self.episode_log_probs.append(log_prob)

    def print_statistics(self):
        if self.action_stats['total'] > 0:
            total = self.action_stats['total']
            
            print(f"  weight adjustment: {self.action_stats['weight']/total:.2%}")
            print(f"  NSarchitecture adjustment: {self.action_stats['ns_arch']/total:.2%}")
            print(f"  Darcyarchitecture adjustment: {self.action_stats['darcy_arch']/total:.2%}")
            print(f"  sampling adjustment: {self.action_stats['sampling']/total:.2%}")

In [ ]:
"""Physics-informed model and training logic"""

class PhysicsInformedNN():
    def __init__(self, X_darcy_t0, X_ns_t0,
                 phi_t0, u1_t0, u2_t0, p_t0,
                 X_darcy_bc, X_ns_bc, X_interface,
                 phi_bc, u1_bc, u2_bc, p_bc,
                 K, nu, g, alpha, s, nf, nd, tau, d, trace,
                 X_darcy_lhs_pool, X_ns_lhs_pool,
                 use_rl=False):

        self.k11 = K[0][0]
        self.k12 = K[0][1]
        self.k21 = K[1][0]
        self.k22 = K[1][1]
        self.nu = nu
        self.g = g
        self.alpha = alpha
        self.s = s
        self.nf1 = nf[0][0]
        self.nf2 = nf[0][1]
        self.np1 = nd[0][0]
        self.np2 = nd[0][1]
        self.tau1 = tau[0][0]
        self.tau2 = tau[0][1]
        self.d = d
        self.trace = trace

        self.domain_classifier = DomainClassifier()

        self.X_darcy_lhs_pool = X_darcy_lhs_pool
        self.X_ns_lhs_pool = X_ns_lhs_pool

        self.N_darcy_base = len(X_darcy_lhs_pool)
        self.N_ns_base = len(X_ns_lhs_pool)

        self.darcy_indices = None
        self.ns_indices = None

        self.N_darcy_refined = 0
        self.N_ns_refined = 0
        self.X_darcy_refined_current = np.empty((0, 3))
        self.X_ns_refined_current = np.empty((0, 3))

        self.X_darcy_bc_fixed = X_darcy_bc
        self.X_ns_bc_fixed = X_ns_bc
        self.X_interface_fixed = X_interface
        self.X_darcy_t0_fixed = X_darcy_t0
        self.X_ns_t0_fixed = X_ns_t0

        self.phi_t0 = torch.tensor(phi_t0, dtype=torch.float32).to(device)
        self.u1_t0 = torch.tensor(u1_t0, dtype=torch.float32).to(device)
        self.u2_t0 = torch.tensor(u2_t0, dtype=torch.float32).to(device)
        self.p_t0 = torch.tensor(p_t0, dtype=torch.float32).to(device)
        self.phi_bc = torch.tensor(phi_bc, dtype=torch.float32).to(device)
        self.u1_bc = torch.tensor(u1_bc, dtype=torch.float32).to(device)
        self.u2_bc = torch.tensor(u2_bc, dtype=torch.float32).to(device)
        self.p_bc = torch.tensor(p_bc, dtype=torch.float32).to(device)

        self.update_residual_points()

        self.arch_config = {
            'ns_depth': 4, 'ns_width': 64,
            'darcy_depth': 4, 'darcy_width': 64,
            'min_depth': 2, 'max_depth': 10,
            'min_units': 5, 'max_units': 256
        }
        self.build_domain_networks()

        self.optimizer_ns = torch.optim.Adam(
            self.ns_network.parameters(),
            lr=0.001,
            weight_decay=1e-5
        )
        self.optimizer_darcy = torch.optim.Adam(
            self.darcy_network.parameters(),
            lr=0.001,
            weight_decay=1e-5
        )

        self.scheduler_ns = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer_ns,
            mode='min',
            factor=0.5,
            patience=500
        )
        self.scheduler_darcy = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer_darcy,
            mode='min',
            factor=0.5,
            patience=500
        )

        self.weights = {
            'darcy': 10.0, 'ns': 10.0, 'interface': 10.0,
            'darcy_bc': 10.0, 'ns_bc': 10.0,
            'darcy_t0': 10.0, 'ns_t0': 10.0
        }
        self._clip_weights()

        self.use_rl = use_rl
        if self.use_rl:
            self.rl_controller = RLController(
                n_actions=28, state_dim=19,
                min_units=self.arch_config['min_units'],
                max_units=self.arch_config['max_units'],
                min_depth=self.arch_config['min_depth'],
                max_depth=self.arch_config['max_depth']
            )
            self.weight_adjust_interval = 200
            self.prev_losses = None
            self.initial_losses = None
            self.last_arch_adjust_step = -10000

        self.iter = 0
        self.adjustment_info = None
        self.sampling_adjustment_info = None
        self.architecture_history = []
        self.record_architecture(0, "Initial")
        self.recent_losses = []
        self.loss_history = []
        self.total_loss_history = []
        self.iteration_history = []
        self.ns_loss_history = []
        self.darcy_loss_history = []
        self.consecutive_bad_steps = 0
        self.loss_explosion_threshold = 50.0
        self.sampling_history = []

        self.cost_stats = {
            'rl_policy_inference_time': 0.0,
            'rl_policy_inference_calls': 0,
            'rl_policy_update_time': 0.0,
            'rl_policy_update_calls': 0
        }
        self.computational_cost = {}

    def build_domain_networks(self):

        ns_depth = self.arch_config['ns_depth']
        ns_width = self.arch_config['ns_width']
        darcy_depth = self.arch_config['darcy_depth']
        darcy_width = self.arch_config['darcy_width']

        ns_layers = []
        ns_layers.append(nn.Linear(3, ns_width))
        ns_layers.append(nn.SiLU())
        for _ in range(ns_depth):
            ns_layers.append(nn.Linear(ns_width, ns_width))
            ns_layers.append(nn.SiLU())
        ns_layers.append(nn.Linear(ns_width, 3))
        self.ns_network = nn.Sequential(*ns_layers).to(device)

        darcy_layers = []
        darcy_layers.append(nn.Linear(3, darcy_width))
        darcy_layers.append(nn.SiLU())
        for _ in range(darcy_depth):
            darcy_layers.append(nn.Linear(darcy_width, darcy_width))
            darcy_layers.append(nn.SiLU())
        darcy_layers.append(nn.Linear(darcy_width, 1))
        self.darcy_network = nn.Sequential(*darcy_layers).to(device)

        self.ns_network.apply(self._init_weights)
        self.darcy_network.apply(self._init_weights)


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def _clip_weights(self):
        for k in self.weights:
            self.weights[k] = max(min(self.weights[k], 100.0), 1.0)

    def forward_ns(self, x, y, t):

        inputs = torch.cat([x, y, t], dim=1)
        outputs = self.ns_network(inputs)
        return outputs[:, 0:1], outputs[:, 1:2], outputs[:, 2:3]

    def forward_darcy(self, x, y, t):

        inputs = torch.cat([x, y, t], dim=1)
        return self.darcy_network(inputs)

    def forward_unified(self, x, y, t):

        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32, device=device).view(-1, 1)
        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.float32, device=device).view(-1, 1)
        if not isinstance(t, torch.Tensor):
            t = torch.tensor(t, dtype=torch.float32, device=device).view(-1, 1)

        n_points = x.shape[0]
        phi = torch.zeros(n_points, 1, device=device)
        u1 = torch.zeros(n_points, 1, device=device)
        u2 = torch.zeros(n_points, 1, device=device)
        p = torch.zeros(n_points, 1, device=device)

        ns_mask = self.domain_classifier.get_ns_mask(x.detach().cpu().numpy(), y.detach().cpu().numpy())
        darcy_mask = self.domain_classifier.get_darcy_mask(x.detach().cpu().numpy(), y.detach().cpu().numpy())
        interface_mask = self.domain_classifier.get_interface_mask(x.detach().cpu().numpy(), y.detach().cpu().numpy())

        if np.any(ns_mask):
            ns_indices = torch.tensor(np.where(ns_mask)[0], device=device, dtype=torch.long)
            if len(ns_indices) > 0:
                x_ns = x[ns_indices]
                y_ns = y[ns_indices]
                t_ns = t[ns_indices]
                u1_ns, u2_ns, p_ns = self.forward_ns(x_ns, y_ns, t_ns)
                u1[ns_indices] = u1_ns
                u2[ns_indices] = u2_ns
                p[ns_indices] = p_ns

        if np.any(darcy_mask):
            darcy_indices = torch.tensor(np.where(darcy_mask)[0], device=device, dtype=torch.long)
            if len(darcy_indices) > 0:
                x_darcy = x[darcy_indices]
                y_darcy = y[darcy_indices]
                t_darcy = t[darcy_indices]
                phi_darcy = self.forward_darcy(x_darcy, y_darcy, t_darcy)
                phi[darcy_indices] = phi_darcy

        if np.any(interface_mask):
            interface_indices = torch.tensor(np.where(interface_mask)[0], device=device, dtype=torch.long)
            if len(interface_indices) > 0:
                x_interface = x[interface_indices]
                y_interface = y[interface_indices]
                t_interface = t[interface_indices]
                u1_interface, u2_interface, p_interface = self.forward_ns(x_interface, y_interface, t_interface)
                phi_interface = self.forward_darcy(x_interface, y_interface, t_interface)
                phi[interface_indices] = phi_interface
                u1[interface_indices] = u1_interface
                u2[interface_indices] = u2_interface
                p[interface_indices] = p_interface

        return phi, u1, u2, p

    def get_architecture_info(self):

        ns_params = sum(p.numel() for p in self.ns_network.parameters())
        darcy_params = sum(p.numel() for p in self.darcy_network.parameters())
        return {
            'ns_depth': self.arch_config['ns_depth'],
            'ns_width': self.arch_config['ns_width'],
            'darcy_depth': self.arch_config['darcy_depth'],
            'darcy_width': self.arch_config['darcy_width'],
            'ns_params': ns_params,
            'darcy_params': darcy_params,
            'total_params': ns_params + darcy_params
        }

    def record_architecture(self, step, action):
        info = self.get_architecture_info()
        info.update({'step': step, 'action': action})
        self.architecture_history.append(info)

    def adjust_architecture(self, command):

        if command is None:
            return False

        success = False

        if command['type'] == 'ns_depth':
            if command['direction'] == 'increase':
                if self.arch_config['ns_depth'] < self.arch_config['max_depth']:
                    self.arch_config['ns_depth'] += 1
                    success = True
            else:
                if self.arch_config['ns_depth'] > self.arch_config['min_depth']:
                    self.arch_config['ns_depth'] -= 1
                    success = True

        elif command['type'] == 'ns_width':
            current_width = self.arch_config['ns_width']
            if command['direction'] == 'increase':
                new_width = int(current_width * 1.1)
                new_width = min(new_width, self.arch_config['max_units'])
                if new_width > current_width:
                    self.arch_config['ns_width'] = new_width
                    success = True
            else:
                new_width = int(current_width * 0.9)
                new_width = max(new_width, self.arch_config['min_units'])
                if new_width < current_width:
                    self.arch_config['ns_width'] = new_width
                    success = True

        elif command['type'] == 'darcy_depth':
            if command['direction'] == 'increase':
                if self.arch_config['darcy_depth'] < self.arch_config['max_depth']:
                    self.arch_config['darcy_depth'] += 1
                    success = True
            else:
                if self.arch_config['darcy_depth'] > self.arch_config['min_depth']:
                    self.arch_config['darcy_depth'] -= 1
                    success = True

        elif command['type'] == 'darcy_width':
            current_width = self.arch_config['darcy_width']
            if command['direction'] == 'increase':
                new_width = int(current_width * 1.1)
                new_width = min(new_width, self.arch_config['max_units'])
                if new_width > current_width:
                    self.arch_config['darcy_width'] = new_width
                    success = True
            else:
                new_width = int(current_width * 0.9)
                new_width = max(new_width, self.arch_config['min_units'])
                if new_width < current_width:
                    self.arch_config['darcy_width'] = new_width
                    success = True

        if success:

            if 'ns' in command['type']:
                old_network = self.ns_network
                self._rebuild_ns_network()
                self._migrate_weights(old_network, self.ns_network)
            else:
                old_network = self.darcy_network
                self._rebuild_darcy_network()
                self._migrate_weights(old_network, self.darcy_network)

            self._reset_optimizer()
            action_desc = f"{command['direction']} {command['type']}"
            self.record_architecture(self.iter, action_desc)
          
            print(f"   NS: depth={self.arch_config['ns_depth']}, width={self.arch_config['ns_width']}")
            print(f"   Darcy: depth={self.arch_config['darcy_depth']}, width={self.arch_config['darcy_width']}")
            return True

        return False

    def _rebuild_ns_network(self):

        ns_layers = []
        ns_layers.append(nn.Linear(3, self.arch_config['ns_width']))
        ns_layers.append(nn.SiLU())
        for _ in range(self.arch_config['ns_depth']):
            ns_layers.append(nn.Linear(self.arch_config['ns_width'], self.arch_config['ns_width']))
            ns_layers.append(nn.SiLU())
        ns_layers.append(nn.Linear(self.arch_config['ns_width'], 3))
        self.ns_network = nn.Sequential(*ns_layers).to(device)
        self.ns_network.apply(self._init_weights)

    def _rebuild_darcy_network(self):

        darcy_layers = []
        darcy_layers.append(nn.Linear(3, self.arch_config['darcy_width']))
        darcy_layers.append(nn.SiLU())
        for _ in range(self.arch_config['darcy_depth']):
            darcy_layers.append(nn.Linear(self.arch_config['darcy_width'], self.arch_config['darcy_width']))
            darcy_layers.append(nn.SiLU())
        darcy_layers.append(nn.Linear(self.arch_config['darcy_width'], 1))
        self.darcy_network = nn.Sequential(*darcy_layers).to(device)
        self.darcy_network.apply(self._init_weights)

    def _migrate_weights(self, old_network, new_network):

        old_layers = [m for m in old_network.modules() if isinstance(m, nn.Linear)]
        new_layers = [m for m in new_network.modules() if isinstance(m, nn.Linear)]

        for old_layer, new_layer in zip(old_layers, new_layers):
            if isinstance(old_layer, nn.Linear) and isinstance(new_layer, nn.Linear):
                min_in = min(new_layer.in_features, old_layer.in_features)
                min_out = min(new_layer.out_features, old_layer.out_features)
                with torch.no_grad():
                    new_layer.weight[:min_out, :min_in] = old_layer.weight[:min_out, :min_in]
                    if new_layer.bias is not None and old_layer.bias is not None:
                        new_layer.bias[:min_out] = old_layer.bias[:min_out]

    def _reset_optimizer(self):

        current_lr_ns = self.optimizer_ns.param_groups[0]['lr']
        current_lr_darcy = self.optimizer_darcy.param_groups[0]['lr']

        self.optimizer_ns = torch.optim.Adam(
            self.ns_network.parameters(),
            lr=current_lr_ns,
            weight_decay=1e-5
        )

        self.optimizer_darcy = torch.optim.Adam(
            self.darcy_network.parameters(),
            lr=current_lr_darcy,
            weight_decay=1e-5
        )

        self.scheduler_ns = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer_ns,
            mode='min',
            factor=0.5,
            patience=500
        )
        self.scheduler_darcy = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer_darcy,
            mode='min',
            factor=0.5,
            patience=500
        )

        self.optimizer_ns.state = defaultdict(dict)
        self.optimizer_darcy.state = defaultdict(dict)

    def update_residual_points(self):

        self.darcy_indices = np.random.choice(
            len(self.X_darcy_lhs_pool),
            self.N_darcy_base,
            replace=False
        )
        selected_darcy = self.X_darcy_lhs_pool[self.darcy_indices]

        self.ns_indices = np.random.choice(
            len(self.X_ns_lhs_pool),
            self.N_ns_base,
            replace=False
        )
        selected_ns = self.X_ns_lhs_pool[self.ns_indices]

        X_f_darcy = np.vstack([
            selected_darcy,
            self.X_darcy_refined_current,
            self.X_darcy_bc_fixed,
            self.X_interface_fixed,
            self.X_darcy_t0_fixed
        ])

        X_f_ns = np.vstack([
            selected_ns,
            self.X_ns_refined_current,
            self.X_ns_bc_fixed,
            self.X_interface_fixed,
            self.X_ns_t0_fixed
        ])

        self.x_darcy = torch.tensor(X_f_darcy[:, 0], dtype=torch.float32, requires_grad=True).to(device)[:, None]
        self.y_darcy = torch.tensor(X_f_darcy[:, 1], dtype=torch.float32, requires_grad=True).to(device)[:, None]
        self.t_darcy = torch.tensor(X_f_darcy[:, 2], dtype=torch.float32, requires_grad=True).to(device)[:, None]

        self.x_ns = torch.tensor(X_f_ns[:, 0], dtype=torch.float32, requires_grad=True).to(device)[:, None]
        self.y_ns = torch.tensor(X_f_ns[:, 1], dtype=torch.float32, requires_grad=True).to(device)[:, None]
        self.t_ns = torch.tensor(X_f_ns[:, 2], dtype=torch.float32, requires_grad=True).to(device)[:, None]

        self.x_interface = torch.tensor(self.X_interface_fixed[:, 0], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.y_interface = torch.tensor(self.X_interface_fixed[:, 1], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.t_interface = torch.tensor(self.X_interface_fixed[:, 2], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]

        self.x_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:, 0], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]
        self.y_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:, 1], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]
        self.t_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:, 2], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]

        self.x_ns_bc = torch.tensor(self.X_ns_bc_fixed[:, 0], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.y_ns_bc = torch.tensor(self.X_ns_bc_fixed[:, 1], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.t_ns_bc = torch.tensor(self.X_ns_bc_fixed[:, 2], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]

        self.x_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:, 0], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]
        self.y_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:, 1], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]
        self.t_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:, 2], dtype=torch.float32, requires_grad=True).to(device)[
                          :, None]

        self.x_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:, 0], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.y_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:, 1], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]
        self.t_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:, 2], dtype=torch.float32, requires_grad=True).to(
            device)[:, None]

    def adjust_sampling_points(self, region, action):

        if region == 'darcy':
            if action == 'add_refined':
                return self._add_refined_points('darcy')
            elif action == 'increase_base':
                return self._adjust_base_points('darcy', increase=True)
            elif action == 'decrease_base':
                return self._adjust_base_points('darcy', increase=False)
        elif region == 'ns':
            if action == 'add_refined':
                return self._add_refined_points('ns')
            elif action == 'increase_base':
                return self._adjust_base_points('ns', increase=True)
            elif action == 'decrease_base':
                return self._adjust_base_points('ns', increase=False)
        return False

    def _add_refined_points(self, region):
        from scipy.stats.qmc import LatinHypercube

        results = self.physics_residuals(return_point_losses=True)

        REFINED_INCREMENT = 1000

        if region == 'darcy':

            if self.N_darcy_refined >= self.rl_controller.max_refined_points:
                print(f" Darcyrefinedsamplinghas reached the upper limit({self.N_darcy_refined})")
                return False

            darcy_residuals = results[7].detach().cpu().numpy()
            n_base = len(self.X_darcy_lhs_pool[self.darcy_indices])
            base_residuals = darcy_residuals[:n_base].flatten()

            current_darcy_points = self.X_darcy_lhs_pool[self.darcy_indices]

            threshold_idx = int(0.8 * len(base_residuals))
            high_loss_indices = np.argsort(base_residuals)[threshold_idx:]
            high_loss_coords = current_darcy_points[high_loss_indices]

            x_min, x_max = high_loss_coords[:, 0].min(), high_loss_coords[:, 0].max()
            y_min, y_max = high_loss_coords[:, 1].min(), high_loss_coords[:, 1].max()
            t_min, t_max = high_loss_coords[:, 2].min(), high_loss_coords[:, 2].max()

            x_range = x_max - x_min
            y_range = y_max - y_min
            t_range = t_max - t_min

            x_min = max(0, x_min - 0.05 * x_range)
            x_max = min(1, x_max + 0.05 * x_range)
            y_min = max(0, y_min - 0.05 * y_range)
            y_max = min(0.25, y_max + 0.05 * y_range)
            t_min = max(0, t_min - 0.05 * t_range)
            t_max = min(1, t_max + 0.05 * t_range)

            n_new = min(REFINED_INCREMENT, self.rl_controller.max_refined_points - self.N_darcy_refined)
            sampler = LatinHypercube(d=3)
            lhs_samples = sampler.random(n_new)

            new_points = lhs_samples * [[x_max - x_min, y_max - y_min, t_max - t_min]] + [[x_min, y_min, t_min]]

            if len(new_points) > 0:
                if len(self.X_darcy_refined_current) == 0:
                    self.X_darcy_refined_current = new_points
                else:
                    self.X_darcy_refined_current = np.vstack([self.X_darcy_refined_current, new_points])
                self.N_darcy_refined = len(self.X_darcy_refined_current)
                
                return True

        else:

            if self.N_ns_refined >= self.rl_controller.max_refined_points:
                print(f" NSrefinedsamplinghas reached the upper limit({self.N_ns_refined})")
                return False

            ns_residuals = results[8].detach().cpu().numpy()
            n_base = len(self.X_ns_lhs_pool[self.ns_indices])
            base_residuals = ns_residuals[:n_base].flatten()

            current_ns_points = self.X_ns_lhs_pool[self.ns_indices]

            threshold_idx = int(0.8 * len(base_residuals))
            high_loss_indices = np.argsort(base_residuals)[threshold_idx:]
            high_loss_coords = current_ns_points[high_loss_indices]

            x_min, x_max = high_loss_coords[:, 0].min(), high_loss_coords[:, 0].max()
            y_min, y_max = high_loss_coords[:, 1].min(), high_loss_coords[:, 1].max()
            t_min, t_max = high_loss_coords[:, 2].min(), high_loss_coords[:, 2].max()

            x_range = x_max - x_min
            y_range = y_max - y_min
            t_range = t_max - t_min

            x_min = max(0, x_min - 0.05 * x_range)
            x_max = min(1, x_max + 0.05 * x_range)
            y_min = max(-0.25, y_min - 0.05 * y_range)
            y_max = min(0, y_max + 0.05 * y_range)
            t_min = max(0, t_min - 0.05 * t_range)
            t_max = min(1, t_max + 0.05 * t_range)

            n_new = min(REFINED_INCREMENT, self.rl_controller.max_refined_points - self.N_ns_refined)
            sampler = LatinHypercube(d=3)
            lhs_samples = sampler.random(n_new)

            new_points = lhs_samples * [[x_max - x_min, y_max - y_min, t_max - t_min]] + [[x_min, y_min, t_min]]

            if len(new_points) > 0:
                if len(self.X_ns_refined_current) == 0:
                    self.X_ns_refined_current = new_points
                else:
                    self.X_ns_refined_current = np.vstack([self.X_ns_refined_current, new_points])
                self.N_ns_refined = len(self.X_ns_refined_current)
               
                return True

        return False

    def _adjust_base_points(self, region, increase):

        if region == 'darcy':
            current_n = self.N_darcy_base
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_darcy_lhs_pool)

            if increase:
                increment = min(int(0.2 * current_n), max_n - current_n)
                if increment > 0:
                    self.N_darcy_base = current_n + increment
                    
                    return True
                else:
                    
                    return False
            else:
                decrement = min(int(0.2 * current_n), current_n - min_n)
                if decrement > 0:
                    self.N_darcy_base = current_n - decrement
                    
                    return True
                else:
                    print(f" Darcybasehas reached the lower limit({current_n})")
                    return False

        else:
            current_n = self.N_ns_base
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_ns_lhs_pool)

            if increase:
                increment = min(int(0.2 * current_n), max_n - current_n)
                if increment > 0:
                    self.N_ns_base = current_n + increment
                    
                    return True
                else:
                    print(f" NSbasehas reached the upper limit({current_n})")
                    return False
            else:
                decrement = min(int(0.2 * current_n), current_n - min_n)
                if decrement > 0:
                    self.N_ns_base = current_n - decrement
                    
                    return True
                else:
                    print(f" NSbasehas reached the lower limit({current_n})")
                    return False

        return False

    def update_loss_history(self, total_loss):

        self.recent_losses.append(total_loss)
        self.loss_history.append(total_loss)

        if len(self.recent_losses) > 100:
            self.recent_losses.pop(0)

        if len(self.loss_history) > 10000:
            self.loss_history.pop(0)

    def physics_residuals(self, return_point_losses=False):

        phi = self.forward_darcy(self.x_darcy, self.y_darcy, self.t_darcy)
        dphi_t = torch.autograd.grad(phi, self.t_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_x = torch.autograd.grad(phi, self.x_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_y = torch.autograd.grad(phi, self.y_darcy, torch.ones_like(phi), create_graph=True)[0]
        up_x = self.k11 * dphi_x
        up_y = self.k22 * dphi_y
        div_up = torch.autograd.grad(up_x, self.x_darcy, grad_outputs=torch.ones_like(up_x), create_graph=True)[0] + \
                 torch.autograd.grad(up_y, self.y_darcy, grad_outputs=torch.ones_like(up_y), create_graph=True)[0]

        f_D = (-torch.pi ** 3 * torch.sin(torch.pi * self.x_darcy) * (-self.y_darcy + torch.cos(torch.pi * (1-self.y_darcy)))-(2-torch.pi * torch.sin(torch.pi * self.x_darcy)) * (-torch.pi ** 2 * torch.cos(torch.pi * (1-self.y_darcy)))) * torch.cos(2*torch.pi * self.t_darcy)-2*torch.pi*(2-torch.pi*torch.sin(torch.pi*self.x_darcy)) * (-self.y_darcy + torch.cos(torch.pi*(1-self.y_darcy)))*torch.sin(2*torch.pi*self.t_darcy)

        darcy_point_residuals = (self.s * dphi_t - (self.g / self.nu) * div_up - f_D) ** 2
        loss_darcy = torch.mean(darcy_point_residuals)

        u1, u2, p = self.forward_ns(self.x_ns, self.y_ns, self.t_ns)

        du1dx = torch.autograd.grad(u1, self.x_ns, torch.ones_like(u1), create_graph=True)[0]
        du1dy = torch.autograd.grad(u1, self.y_ns, torch.ones_like(u1), create_graph=True)[0]
        du2dx = torch.autograd.grad(u2, self.x_ns, torch.ones_like(u2), create_graph=True)[0]
        du2dy = torch.autograd.grad(u2, self.y_ns, torch.ones_like(u2), create_graph=True)[0]

        d2u1dx2 = torch.autograd.grad(du1dx, self.x_ns, grad_outputs=torch.ones_like(du1dx), create_graph=True)[0]
        d2u1dy2 = torch.autograd.grad(du1dy, self.y_ns, grad_outputs=torch.ones_like(du1dy), create_graph=True)[0]
        d2u2dx2 = torch.autograd.grad(du2dx, self.x_ns, grad_outputs=torch.ones_like(du2dx), create_graph=True)[0]
        d2u2dy2 = torch.autograd.grad(du2dy, self.y_ns, grad_outputs=torch.ones_like(du2dy), create_graph=True)[0]

        du1dt = torch.autograd.grad(u1, self.t_ns, grad_outputs=torch.ones_like(u1), create_graph=True)[0]
        du2dt = torch.autograd.grad(u2, self.t_ns, grad_outputs=torch.ones_like(u2), create_graph=True)[0]

        t11 = 2*self.nu*du1dx - p
        t12 = self.nu*(du1dy + du2dx)
        t21 = self.nu*(du1dy + du2dx)
        t22 = 2*self.nu*du2dy - p

        dt11dx = torch.autograd.grad(t11, self.x_ns, torch.ones_like(t11), create_graph=True)[0]
        dt12dy = torch.autograd.grad(t12, self.y_ns, torch.ones_like(t12), create_graph=True)[0]
        dt21dx = torch.autograd.grad(t21, self.x_ns, torch.ones_like(t21), create_graph=True)[0]
        dt22dy = torch.autograd.grad(t22, self.y_ns, torch.ones_like(t22), create_graph=True)[0]

        f1=-2*torch.pi*(self.x_ns**2*self.y_ns**2+torch.exp(-self.y_ns))*torch.sin(2*torch.pi*self.t_ns)+(self.x_ns**2*self.y_ns**2+torch.exp(-self.y_ns))*2*self.x_ns*self.y_ns**2*torch.cos(2*torch.pi*self.t_ns)**2+(-2*self.x_ns*self.y_ns**3/3+2-torch.pi*torch.sin(torch.pi*self.x_ns))*(2*self.x_ns**2*self.y_ns-torch.exp(-self.y_ns))*torch.cos(2*torch.pi*self.t_ns)**2-(self.nu*(2*self.x_ns**2+2*self.y_ns**2+torch.exp(-self.y_ns))-torch.pi**2*torch.cos(torch.pi*self.x_ns)*torch.cos(2*torch.pi*self.y_ns))*torch.cos(2*torch.pi*self.t_ns)
        f2=-2*torch.pi*(-2*self.x_ns*self.y_ns**3/3+2-torch.pi*torch.sin(torch.pi*self.x_ns))*torch.sin(2*torch.pi*self.t_ns)+(self.x_ns**2*self.y_ns**2+torch.exp(-self.y_ns))*(-2*self.y_ns**3/3-torch.pi**2*torch.cos(torch.pi*self.x_ns))*torch.cos(2*torch.pi*self.t_ns)**2+(-2*self.x_ns*self.y_ns**3/3+2-torch.pi*torch.sin(torch.pi*self.x_ns))*(-2*self.x_ns*self.y_ns**2)*torch.cos(2*torch.pi*self.t_ns)**2-(self.nu*(-2*self.x_ns*self.y_ns+torch.pi**3*torch.sin(torch.pi*self.x_ns))-2*torch.pi*(2-torch.pi*torch.sin(torch.pi*self.x_ns))*torch.sin(2*torch.pi*self.y_ns))*torch.cos(2*torch.pi*self.t_ns)

        loss_1 = du1dt + (u1*du1dx + u2*du1dy) - (dt11dx + dt12dy) - f1
        loss_2 = du2dt + (u1*du2dx + u2*du2dy) - (dt21dx + dt22dy) - f2
        loss_3 = du1dx + du2dy

        ns_point_residuals = loss_1**2 + loss_2**2 + loss_3**2
        loss_ns = torch.mean(ns_point_residuals)

        phi_inter = self.forward_darcy(self.x_interface, self.y_interface, self.t_interface)
        u1_inter, u2_inter, p_inter = self.forward_ns(self.x_interface, self.y_interface, self.t_interface)

        dphi_x = torch.autograd.grad(phi_inter, self.x_interface, grad_outputs=torch.ones_like(phi_inter), create_graph=True)[0]
        dphi_y = torch.autograd.grad(phi_inter, self.y_interface, grad_outputs=torch.ones_like(phi_inter), create_graph=True)[0]
        du1dx = torch.autograd.grad(u1_inter, self.x_interface, grad_outputs=torch.ones_like(u1_inter), create_graph=True)[0]
        du1dy = torch.autograd.grad(u1_inter, self.y_interface, grad_outputs=torch.ones_like(u1_inter), create_graph=True)[0]
        du2dx = torch.autograd.grad(u2_inter, self.x_interface, grad_outputs=torch.ones_like(u2_inter), create_graph=True)[0]
        du2dy = torch.autograd.grad(u2_inter, self.y_interface, grad_outputs=torch.ones_like(u2_inter), create_graph=True)[0]

        t11 = 2*self.nu*du1dx - p_inter
        t12 = self.nu*(du1dy + du2dx)
        t21 = self.nu*(du1dy + du2dx)
        t22 = 2*self.nu*du2dy - p_inter

        loss_1 = u1_inter * self.nf1 + u2_inter * self.nf2 - (self.g / self.nu) * (self.k11 * dphi_x * self.np1 + self.k22 * dphi_y * self.np2)
        loss_2 = -self.nf1 * (t11*self.nf1 + t12*self.nf2) - self.nf2 * (t21*self.nf1 + t22*self.nf2) - self.g*phi_inter
        loss_3 = -self.tau1 * (t11*self.nf1 + t12*self.nf2) - self.tau2 * (t21*self.nf1 + t22*self.nf2) - \
                 (self.alpha * self.nu * (self.d ** 0.5) / (self.trace ** 0.5))*(self.tau1 * (u1_inter + (self.g / self.nu) * self.k11 * dphi_x) + \
                  self.tau2 * (u2_inter + (self.g / self.nu) * self.k22 * dphi_y))

        loss_interface = torch.mean((loss_1)**2 + (loss_2)**2 + (loss_3)**2)

        phi_bc_pred = self.forward_darcy(self.x_darcy_bc, self.y_darcy_bc, self.t_darcy_bc)
        loss_darcy_bc = torch.mean((phi_bc_pred - self.phi_bc)**2)

        u1_pred, u2_pred, p_pred = self.forward_ns(self.x_ns_bc, self.y_ns_bc, self.t_ns_bc)
        loss_ns_bc = (torch.mean((u1_pred - self.u1_bc)**2) + torch.mean((u2_pred - self.u2_bc)**2)) + torch.mean((p_pred - self.p_bc)**2)

        phi_pred = self.forward_darcy(self.x_darcy_t0, self.y_darcy_t0, self.t_darcy_t0)
        loss_darcy_t0 = torch.mean((phi_pred - self.phi_t0)**2)

        u1_pred, u2_pred, p_pred = self.forward_ns(self.x_ns_t0, self.y_ns_t0, self.t_ns_t0)
        loss_ns_t0 = torch.mean((u1_pred - self.u1_t0)**2) + torch.mean((u2_pred - self.u2_t0)**2) + torch.mean((p_pred - self.p_t0)**2)

        if return_point_losses:
            return (
            loss_darcy, loss_ns, loss_interface, loss_darcy_bc, loss_ns_bc, loss_darcy_t0, loss_ns_t0,
            darcy_point_residuals, ns_point_residuals)
        else:
            return loss_darcy, loss_ns, loss_interface, loss_darcy_bc, loss_ns_bc, loss_darcy_t0, loss_ns_t0

    def loss_func(self, training=True):
        if training:

            self.optimizer_ns.zero_grad()
            self.optimizer_darcy.zero_grad()

        losses = self.physics_residuals(return_point_losses=False)

        self.current_losses = losses

        if self.initial_losses is None:
            self.initial_losses = [l.item() for l in losses]

        loss_darcy = losses[0]
        loss_ns = losses[1]
        loss_interface = losses[2]
        loss_darcy_bc = losses[3]
        loss_ns_bc = losses[4]
        loss_darcy_t0 = losses[5]
        loss_ns_t0 = losses[6]

        if self.use_rl and (self.iter % self.weight_adjust_interval == 0):
            arch_info = self.get_architecture_info()
            current_losses = [l.item() for l in losses]

            rl_decision_start_time = cost_timer()
            state = self.rl_controller.get_enhanced_state(
                current_losses,
                arch_info['ns_depth'],
                arch_info['ns_width'],
                arch_info['darcy_depth'],
                arch_info['darcy_width'],
                self.loss_history,
                self.N_darcy_base,
                self.N_ns_base,
                self.N_darcy_refined,
                self.N_ns_refined
            )

            action_id, log_prob = self.rl_controller.select_action(state)
            self.cost_stats['rl_policy_inference_time'] += cost_timer() - rl_decision_start_time
            self.cost_stats['rl_policy_inference_calls'] += 1

            if action_id < 14:
                action_map = {0: +10, 1: -10}
                adjust_factor = action_map[action_id % 2]
                target_loss_idx = action_id // 2
                loss_keys = ['darcy', 'ns', 'interface', 'darcy_bc', 'ns_bc', 'darcy_t0', 'ns_t0']
                selected_key = loss_keys[target_loss_idx]
                self.weights[selected_key] += adjust_factor
                self._clip_weights()
                self.rl_controller.add_log_prob(log_prob)
                self.prev_losses = current_losses

                if self.prev_losses is not None:
                    reward = calculate_reward(current_losses, self.prev_losses, self.initial_losses)
                    self.rl_controller.add_reward(reward)

            elif action_id < 22:
                command = self.rl_controller.create_architecture_command(action_id)
                self.adjustment_info = {
                    'step': self.iter,
                    'pre_losses': current_losses,
                    'arch_action': action_id,
                    'log_prob': log_prob,
                    'command': command
                }
                self.rl_controller.add_log_prob(log_prob)

                if self.adjust_architecture(command):
                    self.last_arch_adjust_step = self.iter
                    action_desc = f"{command['direction']} {command['type']}"
                    self.adjustment_info['action_desc'] = action_desc

            else:
                command = self.rl_controller.create_sampling_command(action_id)

                if command:
                    self.sampling_adjustment_info = {
                        'step': self.iter,
                        'pre_losses': current_losses,
                        'log_prob': log_prob,
                        'command': command
                    }
                    self.rl_controller.add_log_prob(log_prob)

                    success = self.adjust_sampling_points(command['region'], command['action'])

                    if success:

                        self.update_residual_points()

                        self.sampling_history.append({
                            'step': self.iter,
                            'command': command,
                            'darcy_base': self.N_darcy_base,
                            'ns_base': self.N_ns_base,
                            'darcy_refined': self.N_darcy_refined,
                            'ns_refined': self.N_ns_refined
                        })
                    else:
                        self.sampling_adjustment_info = None

        if self.adjustment_info and (self.iter - self.adjustment_info['step']) >= 200:
            current_losses = [l.item() for l in losses]
            pre_losses = self.adjustment_info['pre_losses']
            action_desc = self.adjustment_info.get('action_desc', 'unknown')

            arch_info = self.get_architecture_info()
            arch_reward = calculate_arch_reward(
                pre_losses, current_losses,
                arch_info['total_params'],
                action_desc,
                None, None
            )

            self.rl_controller.add_reward(arch_reward)

            total_improvement = (sum(pre_losses) - sum(current_losses)) / (sum(pre_losses) + 1e-9)
           
            print(f"   loss: {sum(pre_losses):.2e} -> {sum(current_losses):.2e}")
            print(f"   improvement: {total_improvement:+.2%}")
            print(f"   reward: {arch_reward:+.1f}")

            self.adjustment_info = None

        if hasattr(self, 'sampling_adjustment_info') and self.sampling_adjustment_info:
            steps_elapsed = self.iter - self.sampling_adjustment_info['step']

            if steps_elapsed == 200:
                current_losses_val = [l.item() for l in losses]
                pre_losses_val = self.sampling_adjustment_info['pre_losses']

                sampling_reward = calculate_reward(
                    current_losses_val,
                    pre_losses_val,
                    self.initial_losses
                )

                self.rl_controller.add_reward(sampling_reward)
                self.sampling_adjustment_info = None

        darcy_network_loss = (
            self.weights['darcy'] * loss_darcy +
            self.weights['darcy_bc'] * loss_darcy_bc +
            self.weights['darcy_t0'] * loss_darcy_t0 +
            self.weights['interface'] * loss_interface
        )

        ns_network_loss = (
            self.weights['ns'] * loss_ns +
            self.weights['ns_bc'] * loss_ns_bc +
            self.weights['ns_t0'] * loss_ns_t0 +
            self.weights['interface'] * loss_interface
        )

        total_loss = (
            self.weights['darcy'] * loss_darcy +
            self.weights['ns'] * loss_ns +
            self.weights['interface'] * loss_interface +
            self.weights['darcy_bc'] * loss_darcy_bc +
            self.weights['ns_bc'] * loss_ns_bc +
            self.weights['darcy_t0'] * loss_darcy_t0 +
            self.weights['ns_t0'] * loss_ns_t0
        )

        if training:

            darcy_network_loss.backward(retain_graph=True)
            ns_network_loss.backward()

            torch.nn.utils.clip_grad_norm_(self.darcy_network.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.ns_network.parameters(), max_norm=1.0)

            self.optimizer_darcy.step()
            self.optimizer_ns.step()

        self.update_loss_history(total_loss.item())

        if self.iter % 10 == 0:
            self.total_loss_history.append(total_loss.item())
            self.ns_loss_history.append(ns_network_loss.item())
            self.darcy_loss_history.append(darcy_network_loss.item())
            self.iteration_history.append(self.iter)

        current_total_loss = total_loss.item()
        if current_total_loss > 100 and hasattr(self, 'rl_controller') and self.use_rl:
            if self.iter % 100 == 0:
                print(f" Current loss is large({current_total_loss:.2e})")

        if current_total_loss > self.loss_explosion_threshold:
            self.consecutive_bad_steps += 1
        else:
            self.consecutive_bad_steps = 0

        self._clip_weights()
        self.iter += 1

        if self.iter % 100 == 0:
            self.scheduler_ns.step(ns_network_loss.item())
            self.scheduler_darcy.step(darcy_network_loss.item())

        if self.use_rl and (self.iter % 500 == 0) and (len(self.rl_controller.episode_rewards) > 0):
            scaled_rewards = [r * 0.1 for r in self.rl_controller.episode_rewards]
            self.rl_controller.episode_rewards = scaled_rewards
            rl_update_start_time = cost_timer()
            self.rl_controller.update_policy()
            self.cost_stats['rl_policy_update_time'] += cost_timer() - rl_update_start_time
            self.cost_stats['rl_policy_update_calls'] += 1

        if self.iter % 100 == 0:
            arch_info = self.get_architecture_info()
            total_darcy = self.N_darcy_base + self.N_darcy_refined
            total_ns = self.N_ns_base + self.N_ns_refined
            print(
                f'Iter {self.iter} | Total Loss: {total_loss.item():.2e} | '
                f'NS Loss: {ns_network_loss.item():.2e} | '
                f'Darcy Loss: {darcy_network_loss.item():.2e}'
            )
            print(
                f'   architecture: NS [{arch_info["ns_depth"]}x{arch_info["ns_width"]}] | '
                f'Darcy [{arch_info["darcy_depth"]}x{arch_info["darcy_width"]}] | '
                f'Darcysampling: {self.N_darcy_base}+{self.N_darcy_refined}={total_darcy}, '
                f'NSsampling: {self.N_ns_base}+{self.N_ns_refined}={total_ns} | '
                f'LR_NS: {self.optimizer_ns.param_groups[0]["lr"]:.1e}, '
                f'LR_Darcy: {self.optimizer_darcy.param_groups[0]["lr"]:.1e}'
            )
            print(
                f'   loss: Darcy: {loss_darcy.item():.2e} (w={self.weights["darcy"]:.2f}), '
                f'NS: {loss_ns.item():.2e} (w={self.weights["ns"]:.2f}), '
                f'Interface: {loss_interface.item():.2e} (w={self.weights["interface"]:.2f}), '
                f'Darcy_BC: {loss_darcy_bc.item():.2e} (w={self.weights["darcy_bc"]:.2f}), '
                f'NS_BC: {loss_ns_bc.item():.2e} (w={self.weights["ns_bc"]:.2f}), '
                f'Darcy_T0: {loss_darcy_t0.item():.2e} (w={self.weights["darcy_t0"]:.2f}), '
                f'NS_T0: {loss_ns_t0.item():.2e} (w={self.weights["ns_t0"]:.2f})'
            )

            if self.use_rl:
                self.rl_controller.print_statistics()

        return total_loss

    def train(self, stage1_epochs=20000, stage2_epochs=10000):


        training_start_time = time.time()

        initial_config = {
            'ns_depth': self.arch_config['ns_depth'],
            'ns_width': self.arch_config['ns_width'],
            'darcy_depth': self.arch_config['darcy_depth'],
            'darcy_width': self.arch_config['darcy_width'],
            'weights': self.weights.copy(),
            'N_darcy_base': self.N_darcy_base,
            'N_ns_base': self.N_ns_base,
            'N_darcy_refined': self.N_darcy_refined,
            'N_ns_refined': self.N_ns_refined
        }


        reset_cuda_peak_memory()
        stage1_start_time = time.time()
        self.ns_network.train()
        self.darcy_network.train()
        stage1_losses = []

        for epoch in range(stage1_epochs):

            loss = self.loss_func(training=True)

            stage1_losses.append(loss.item())

            if epoch % 1000 == 0:
                arch_info = self.get_architecture_info()
                elapsed_time = time.time() - stage1_start_time
                total_darcy = self.N_darcy_base + self.N_darcy_refined
                total_ns = self.N_ns_base + self.N_ns_refined

                current_losses = self.current_losses

                print(
                    f'\nStage 1 -  {epoch}/{stage1_epochs} | Total Loss: {loss.item():.2e} | '
                    f'NS: [{arch_info["ns_depth"]}x{arch_info["ns_width"]}], '
                    f'Darcy: [{arch_info["darcy_depth"]}x{arch_info["darcy_width"]}] | '
                    f'Darcysampling: {self.N_darcy_base}+{self.N_darcy_refined}={total_darcy}, '
                    f'NSsampling: {self.N_ns_base}+{self.N_ns_refined}={total_ns} | '
                    f'LR_NS: {self.optimizer_ns.param_groups[0]["lr"]:.1e}, '
                    f'LR_Darcy: {self.optimizer_darcy.param_groups[0]["lr"]:.1e} | '
                    f'elapsed time: {elapsed_time:.1f}s | '
                    f'Darcy: {current_losses[0].item():.2e} (w={self.weights["darcy"]:.2f}), '
                    f'NS: {current_losses[1].item():.2e} (w={self.weights["ns"]:.2f}), '
                    f'Interface: {current_losses[2].item():.2e} (w={self.weights["interface"]:.2f}), '
                    f'Darcy_BC: {current_losses[3].item():.2e} (w={self.weights["darcy_bc"]:.2f}), '
                    f'NS_BC: {current_losses[4].item():.2e} (w={self.weights["ns_bc"]:.2f}), '
                    f'Darcy_T0: {current_losses[5].item():.2e} (w={self.weights["darcy_t0"]:.2f}), '
                    f'NS_T0: {current_losses[6].item():.2e} (w={self.weights["ns_t0"]:.2f})'
                )

        sync_cuda_if_available()
        stage1_end_time = time.time()
        stage1_duration = stage1_end_time - stage1_start_time
        stage1_memory = get_cuda_memory_summary()

        stage1_final_config = {
            'ns_depth': self.arch_config['ns_depth'],
            'ns_width': self.arch_config['ns_width'],
            'darcy_depth': self.arch_config['darcy_depth'],
            'darcy_width': self.arch_config['darcy_width'],
            'weights': self.weights.copy(),
            'N_darcy_base': self.N_darcy_base,
            'N_ns_base': self.N_ns_base,
            'N_darcy_refined': self.N_darcy_refined,
            'N_ns_refined': self.N_ns_refined,
            'final_loss': stage1_losses[-1],
            'training_time': stage1_duration
        }

        print(f"\n Stage 1completed (elapsed time: {stage1_duration:.1f}s, {stage1_duration / 60:.1f}min):")
        print(f"   InitialNS: depth={initial_config['ns_depth']}, width={initial_config['ns_width']}")
        print(f"   FinalNS: depth={stage1_final_config['ns_depth']}, width={stage1_final_config['ns_width']}")
        print(f"   InitialDarcy: depth={initial_config['darcy_depth']}, width={initial_config['darcy_width']}")
        print(f"   FinalDarcy: depth={stage1_final_config['darcy_depth']}, width={stage1_final_config['darcy_width']}")
        print(
            f"   Darcysampling: base{stage1_final_config['N_darcy_base']} + refined{stage1_final_config['N_darcy_refined']} = {stage1_final_config['N_darcy_base'] + stage1_final_config['N_darcy_refined']}")
        print(
            f"   NSsampling: base{stage1_final_config['N_ns_base']} + refined{stage1_final_config['N_ns_refined']} = {stage1_final_config['N_ns_base'] + stage1_final_config['N_ns_refined']}")
        print(f"   Finalloss: {stage1_final_config['final_loss']:.2e}")

        reset_cuda_peak_memory()
        stage2_start_time = time.time()

        original_use_rl = self.use_rl
        self.use_rl = False

        stage2_losses = []

        for epoch in range(stage2_epochs):

            losses = self.physics_residuals()

            loss_darcy = losses[0]
            loss_ns = losses[1]
            loss_interface = losses[2]
            loss_darcy_bc = losses[3]
            loss_ns_bc = losses[4]
            loss_darcy_t0 = losses[5]
            loss_ns_t0 = losses[6]

            darcy_network_loss = (
                self.weights['darcy'] * loss_darcy +
                self.weights['darcy_bc'] * loss_darcy_bc +
                self.weights['darcy_t0'] * loss_darcy_t0 +
                self.weights['interface'] * loss_interface
            )

            ns_network_loss = (
                self.weights['ns'] * loss_ns +
                self.weights['ns_bc'] * loss_ns_bc +
                self.weights['ns_t0'] * loss_ns_t0 +
                self.weights['interface'] * loss_interface
            )

            total_loss = (
                self.weights['darcy'] * loss_darcy +
                self.weights['ns'] * loss_ns +
                self.weights['interface'] * loss_interface +
                self.weights['darcy_bc'] * loss_darcy_bc +
                self.weights['ns_bc'] * loss_ns_bc +
                self.weights['darcy_t0'] * loss_darcy_t0 +
                self.weights['ns_t0'] * loss_ns_t0
            )

            self.optimizer_darcy.zero_grad()
            self.optimizer_ns.zero_grad()

            darcy_network_loss.backward(retain_graph=True)
            ns_network_loss.backward()

            torch.nn.utils.clip_grad_norm_(self.darcy_network.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.ns_network.parameters(), max_norm=1.0)

            self.optimizer_darcy.step()
            self.optimizer_ns.step()

            self.update_loss_history(total_loss.item())
            if self.iter % 10 == 0:
                self.total_loss_history.append(total_loss.item())
                self.ns_loss_history.append(ns_network_loss.item())
                self.darcy_loss_history.append(darcy_network_loss.item())
                self.iteration_history.append(self.iter)

            current_loss = total_loss.item()
            stage2_losses.append(current_loss)

            if epoch % 100 == 0:
                self.scheduler_ns.step(ns_network_loss.item())
                self.scheduler_darcy.step(darcy_network_loss.item())

            if epoch % 1000 == 0:
                elapsed_time = time.time() - stage2_start_time
                arch_info = self.get_architecture_info()
                total_darcy = self.N_darcy_base + self.N_darcy_refined
                total_ns = self.N_ns_base + self.N_ns_refined
                print(
                    f'\nStage 2 -  {epoch}/{stage2_epochs} | Total Loss: {current_loss:.2e} | '
                    f'NS Loss: {ns_network_loss.item():.2e} | '
                    f'Darcy Loss: {darcy_network_loss.item():.2e} | '
                    f'NS: [{arch_info["ns_depth"]}x{arch_info["ns_width"]}], '
                    f'Darcy: [{arch_info["darcy_depth"]}x{arch_info["darcy_width"]}] | '
                    f'Darcysampling: {self.N_darcy_base}+{self.N_darcy_refined}={total_darcy}, '
                    f'NSsampling: {self.N_ns_base}+{self.N_ns_refined}={total_ns} | '
                    f'LR_NS: {self.optimizer_ns.param_groups[0]["lr"]:.1e}, '
                    f'LR_Darcy: {self.optimizer_darcy.param_groups[0]["lr"]:.1e} | '
                    f'elapsed time: {elapsed_time:.1f}s'
                )

            self.iter += 1

        sync_cuda_if_available()
        stage2_end_time = time.time()
        stage2_duration = stage2_end_time - stage2_start_time
        stage2_memory = get_cuda_memory_summary()

        final_iter = self.iter - 1
        if final_iter not in self.iteration_history:
            if len(stage2_losses) > 0:

                losses = self.physics_residuals()
                loss_darcy = losses[0]
                loss_ns = losses[1]
                loss_interface = losses[2]
                loss_darcy_bc = losses[3]
                loss_ns_bc = losses[4]
                loss_darcy_t0 = losses[5]
                loss_ns_t0 = losses[6]

                darcy_network_loss = (
                    self.weights['darcy'] * loss_darcy +
                    self.weights['darcy_bc'] * loss_darcy_bc +
                    self.weights['darcy_t0'] * loss_darcy_t0 +
                    self.weights['interface'] * loss_interface
                )

                ns_network_loss = (
                    self.weights['ns'] * loss_ns +
                    self.weights['ns_bc'] * loss_ns_bc +
                    self.weights['ns_t0'] * loss_ns_t0 +
                    self.weights['interface'] * loss_interface
                )

                self.total_loss_history.append(stage2_losses[-1])
                self.ns_loss_history.append(ns_network_loss.item())
                self.darcy_loss_history.append(darcy_network_loss.item())
                self.iteration_history.append(final_iter)
                print(f" Supplement the final step record: iter={final_iter}, loss={stage2_losses[-1]:.2e}")

        self.use_rl = original_use_rl

        total_training_time = time.time() - training_start_time

        print(f"\n Two-stage training completed:")
        print(f"   Stage 1elapsed time: {stage1_duration:.1f}s ({stage1_duration / 60:.1f}min)")
        print(f"   Stage 2elapsed time: {stage2_duration:.1f}s ({stage2_duration / 60:.1f}min)")
        print(f"   Total training time: {total_training_time:.1f}s ({total_training_time / 60:.1f}min)")
        print(f"   Stage 1Finalloss: {stage1_final_config['final_loss']:.2e}")
        print(f"   Stage 2Finalloss: {stage2_losses[-1]:.2e}")
        print(f"   FinalNSarchitecture: depth={self.arch_config['ns_depth']}, width={self.arch_config['ns_width']}")
        print(f"   FinalDarcyarchitecture: depth={self.arch_config['darcy_depth']}, width={self.arch_config['darcy_width']}")

        self.computational_cost = self.build_computational_cost_report(
            stage1_duration=stage1_duration,
            stage2_duration=stage2_duration,
            total_training_time=total_training_time,
            stage1_memory=stage1_memory,
            stage2_memory=stage2_memory,
            stage1_epochs=stage1_epochs,
            stage2_epochs=stage2_epochs
        )
        write_computational_cost_report(self.computational_cost, prefix=COST_REPORT_PREFIX)

        return {
            'stage1_losses': stage1_losses,
            'stage2_losses': stage2_losses,
            'stage1_config': stage1_final_config,
            'stage1_time': stage1_duration,
            'stage2_time': stage2_duration,
            'total_time': total_training_time,
            'computational_cost': self.computational_cost
        }

    def build_computational_cost_report(self, stage1_duration, stage2_duration, total_training_time,
                                        stage1_memory, stage2_memory, stage1_epochs, stage2_epochs):

        ns_params = count_trainable_parameters(self.ns_network)
        darcy_params = count_trainable_parameters(self.darcy_network)
        trainable_params = int(ns_params + darcy_params)
        final_training_points = int(
            self.N_darcy_base + self.N_darcy_refined +
            self.N_ns_base + self.N_ns_refined +
            len(self.X_darcy_bc_fixed) + len(self.X_ns_bc_fixed) +
            len(self.X_interface_fixed) + len(self.X_darcy_t0_fixed) + len(self.X_ns_t0_fixed)
        )
        peak_allocated = None
        peak_reserved = None
        if stage1_memory.get('cuda_available') or stage2_memory.get('cuda_available'):
            peak_allocated = max(stage1_memory.get('peak_allocated_mb') or 0, stage2_memory.get('peak_allocated_mb') or 0)
            peak_reserved = max(stage1_memory.get('peak_reserved_mb') or 0, stage2_memory.get('peak_reserved_mb') or 0)

        rl_inference = self.cost_stats['rl_policy_inference_time']
        rl_update = self.cost_stats['rl_policy_update_time']
        rl_total = rl_inference + rl_update

        report = {
            'problem': 'Coupled Navier-Stokes-Darcy system',
            'method': 'Architecture III (Fully Decoupled Dual-Network, Dual Optimizer, Auto-PINNs)',
            'hardware': get_hardware_info(),
            'model': {
                'network_type': 'Fully decoupled dual-network with dual optimizers',
                'ns_architecture_depth': int(self.arch_config['ns_depth']),
                'ns_architecture_width': int(self.arch_config['ns_width']),
                'darcy_architecture_depth': int(self.arch_config['darcy_depth']),
                'darcy_architecture_width': int(self.arch_config['darcy_width']),
                'ns_trainable_parameters': int(ns_params),
                'darcy_trainable_parameters': int(darcy_params),
                'trainable_parameters': trainable_params,
                'cost_note': 'FLOPs are not reported because exact FLOP counting for PINNs with automatic differentiation and dynamic sampling is implementation-dependent. GPU-hours are used instead.'
            },
            'training': {
                'stage1_epochs': int(stage1_epochs),
                'stage2_epochs': int(stage2_epochs),
                'stage1_time_s': float(stage1_duration),
                'stage2_time_s': float(stage2_duration),
                'total_training_time_s': float(total_training_time),
                'gpu_hours': float(total_training_time / 3600.0)
            },
            'sampling': {
                'final_darcy_base_points': int(self.N_darcy_base),
                'final_darcy_refined_points': int(self.N_darcy_refined),
                'final_ns_base_points': int(self.N_ns_base),
                'final_ns_refined_points': int(self.N_ns_refined),
                'final_training_points_approx': final_training_points,
                'darcy_bc_points': int(len(self.X_darcy_bc_fixed)),
                'ns_bc_points': int(len(self.X_ns_bc_fixed)),
                'interface_points': int(len(self.X_interface_fixed)),
                'darcy_t0_points': int(len(self.X_darcy_t0_fixed)),
                'ns_t0_points': int(len(self.X_ns_t0_fixed))
            },
            'memory': {
                'stage1_cuda_memory': stage1_memory,
                'stage2_cuda_memory': stage2_memory,
                'training_peak_allocated_mb': peak_allocated,
                'training_peak_reserved_mb': peak_reserved,
                'cpu_rss_mb_at_report': get_cpu_memory_mb()
            },
            'rl_overhead': {
                'policy_inference_time_s': float(rl_inference),
                'policy_inference_calls': int(self.cost_stats['rl_policy_inference_calls']),
                'policy_update_time_s': float(rl_update),
                'policy_update_calls': int(self.cost_stats['rl_policy_update_calls']),
                'total_measured_overhead_time_s': float(rl_total),
                'overhead_fraction_of_training_time': float(rl_total / total_training_time) if total_training_time > 0 else None,
                'overhead_percent_of_training_time': float(100.0 * rl_total / total_training_time) if total_training_time > 0 else None
            },
            'inference': {}
        }
        return report

    def finalize_computational_cost_report(self, prediction_duration=None, prediction_points=None,
                                           prefix=COST_REPORT_PREFIX):

        if not getattr(self, 'computational_cost', None):
            ns_params = count_trainable_parameters(self.ns_network)
            darcy_params = count_trainable_parameters(self.darcy_network)
            self.computational_cost = {
                'problem': 'Coupled Navier-Stokes-Darcy system',
                'method': 'Architecture III (Fully Decoupled Dual-Network, Dual Optimizer, Auto-PINNs)',
                'hardware': get_hardware_info(),
                'model': {
                    'network_type': 'Fully decoupled dual-network with dual optimizers',
                    'ns_architecture_depth': int(self.arch_config['ns_depth']),
                    'ns_architecture_width': int(self.arch_config['ns_width']),
                    'darcy_architecture_depth': int(self.arch_config['darcy_depth']),
                    'darcy_architecture_width': int(self.arch_config['darcy_width']),
                    'ns_trainable_parameters': int(ns_params),
                    'darcy_trainable_parameters': int(darcy_params),
                    'trainable_parameters': int(ns_params + darcy_params),
                    'cost_note': 'FLOPs are not reported because exact FLOP counting for PINNs with automatic differentiation and dynamic sampling is implementation-dependent. GPU-hours are used instead.'
                },
                'training': {}, 'memory': {}, 'rl_overhead': {}, 'sampling': {}
            }

        inference = self.computational_cost.setdefault('inference', {})
        if prediction_duration is not None:
            inference['prediction_time_s'] = float(prediction_duration)
        if prediction_points is not None:
            inference['prediction_points'] = int(prediction_points)
            if prediction_duration is not None and prediction_points > 0:
                inference['prediction_time_per_point_s'] = float(prediction_duration / prediction_points)
        if 'training' in self.computational_cost and prediction_duration is not None:
            training_time = self.computational_cost['training'].get('total_training_time_s')
            if training_time is not None:
                self.computational_cost['total_time_including_prediction_s'] = float(training_time + prediction_duration)
        paths = write_computational_cost_report(self.computational_cost, prefix=prefix)
        return paths

    def predict(self, XYT):

        X = XYT[:, 0:1]
        Y = XYT[:, 1:2]
        T = XYT[:, 2:3]
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        Y_tensor = torch.tensor(Y, dtype=torch.float32).to(device)
        T_tensor = torch.tensor(T, dtype=torch.float32).to(device)
        with torch.no_grad():
            phi, u1, u2, p = self.forward_unified(X_tensor, Y_tensor, T_tensor)
        return phi.cpu().numpy(), u1.cpu().numpy(), u2.cpu().numpy(), p.cpu().numpy()
    


In [ ]:
"""Physical parameters and data preparation"""


K = [[1.0, 0.0], [0.0, 1.0]]
nu = 1.0
g = 1.0
alpha = 1.0
s = 1.0
nf = [[0.0, 1.0]]
nd = [[0.0, -1.0]]
tau = [[1.0, 0.0]]
d = 2.0
trace = 2.0

N_D_t0 = 500
N_S_t0 = 500
N_u_darcy_bc = 500
N_u_ns_bc = 500
N_u_interface = 500

sampler_darcy_lhs = LatinHypercube(d=3)
X_darcy_lhs_pool = sampler_darcy_lhs.random(5000) * [1, 0.75, 1] + [0, 0, 0]

sampler_ns_lhs = LatinHypercube(d=3)
X_ns_lhs_pool = sampler_ns_lhs.random(5000) * [1, 0.25, 1] + [0, -0.25, 0]

print(" LHSsampling pointspool生成completed:")
print(f"   Darcypool: {X_darcy_lhs_pool.shape}")
print(f"   NSpool: {X_ns_lhs_pool.shape}")

data_phi = scipy.io.loadmat('./open_source_assets/data/phi.mat')
data_u1 = scipy.io.loadmat('./open_source_assets/data/u1.mat')
data_u2 = scipy.io.loadmat('./open_source_assets/data/u2.mat')
data_p = scipy.io.loadmat('./open_source_assets/data/p.mat')

X_D_2d = data_phi['X_D']
Y_D_2d = data_phi['Y_D']
X_S_2d = data_u1['X_S']
Y_S_2d = data_u1['Y_S']

X_D = X_D_2d[0, :]
Y_D = Y_D_2d[:, 0]
X_S = X_S_2d[0, :]
Y_S = Y_S_2d[:, 0]

T = data_phi['t_values'].flatten()[:, None]

Phi = np.real(data_phi['phi'])
U1 = np.real(data_u1['u1'])
U2 = np.real(data_u2['u2'])
P = np.real(data_p['p'])

X_D_grid, Y_D_grid, T_D_grid = np.meshgrid(X_D, Y_D, T)
X_S_grid, Y_S_grid, T_S_grid = np.meshgrid(X_S, Y_S, T)

X_darcy_initial = X_D_grid[:, :, 0].reshape(-1, 1)
Y_darcy_initial = Y_D_grid[:, :, 0].reshape(-1, 1)
T_darcy_initial = np.zeros_like(X_darcy_initial)
X_darcy_t0 = np.hstack([X_darcy_initial, Y_darcy_initial, T_darcy_initial])
phi_t0 = Phi[:, :, 0].reshape(-1, 1)

X_ns_initial = X_S_grid[:, :, 0].reshape(-1, 1)
Y_ns_initial = Y_S_grid[:, :, 0].reshape(-1, 1)
T_ns_initial = np.zeros_like(X_ns_initial)
X_ns_t0 = np.hstack([X_ns_initial, Y_ns_initial, T_ns_initial])
u1_t0 = U1[:, :, 0].reshape(-1, 1)
u2_t0 = U2[:, :, 0].reshape(-1, 1)
p_t0 = P[:, :, 0].reshape(-1, 1)

xx_darcy_left = np.hstack(
    [X_D_grid[:, 0, :].reshape(-1, 1), Y_D_grid[:, 0, :].reshape(-1, 1), T_D_grid[:, 0, :].reshape(-1, 1)])
phi_darcy_left = Phi[:, 0, :].reshape(-1, 1)

xx_darcy_right = np.hstack(
    [X_D_grid[:, -1, :].reshape(-1, 1), Y_D_grid[:, -1, :].reshape(-1, 1), T_D_grid[:, -1, :].reshape(-1, 1)])
phi_darcy_right = Phi[:, -1, :].reshape(-1, 1)

xx_darcy_top = np.hstack(
    [X_D_grid[-1, :, :].reshape(-1, 1), Y_D_grid[-1, :, :].reshape(-1, 1), T_D_grid[-1, :, :].reshape(-1, 1)])
phi_darcy_top = Phi[-1, :, :].reshape(-1, 1)

X_train_darcy_bc = np.vstack([xx_darcy_left, xx_darcy_right, xx_darcy_top])
phi_bc = np.vstack([phi_darcy_left, phi_darcy_right, phi_darcy_top])

xx_ns_left = np.hstack(
    [X_S_grid[:, 0, :].reshape(-1, 1), Y_S_grid[:, 0, :].reshape(-1, 1), T_S_grid[:, 0, :].reshape(-1, 1)])
u1_ns_left = U1[:, 0, :].reshape(-1, 1)
u2_ns_left = U2[:, 0, :].reshape(-1, 1)
p_ns_left = P[:, 0, :].reshape(-1, 1)

xx_ns_right = np.hstack(
    [X_S_grid[:, -1, :].reshape(-1, 1), Y_S_grid[:, -1, :].reshape(-1, 1), T_S_grid[:, -1, :].reshape(-1, 1)])
u1_ns_right = U1[:, -1, :].reshape(-1, 1)
u2_ns_right = U2[:, -1, :].reshape(-1, 1)
p_ns_right = P[:, -1, :].reshape(-1, 1)

xx_ns_bottom = np.hstack(
    [X_S_grid[0, :, :].reshape(-1, 1), Y_S_grid[0, :, :].reshape(-1, 1), T_S_grid[0, :, :].reshape(-1, 1)])
u1_ns_bottom = U1[0, :, :].reshape(-1, 1)
u2_ns_bottom = U2[0, :, :].reshape(-1, 1)
p_ns_bottom = P[0, :, :].reshape(-1, 1)

X_train_ns_bc = np.vstack([xx_ns_left, xx_ns_right, xx_ns_bottom])
u1_bc = np.vstack([u1_ns_left, u1_ns_right, u1_ns_bottom])
u2_bc = np.vstack([u2_ns_left, u2_ns_right, u2_ns_bottom])
p_bc = np.vstack([p_ns_left, p_ns_right, p_ns_bottom])

X_train_interface = np.hstack(
    [X_S_grid[-1, :, :].reshape(-1, 1), Y_S_grid[-1, :, :].reshape(-1, 1), T_S_grid[-1, :, :].reshape(-1, 1)])

idx_D_t0 = np.random.choice(X_darcy_t0.shape[0], N_D_t0, replace=False)
X_darcy_t0 = X_darcy_t0[idx_D_t0, :]
phi_t0 = phi_t0[idx_D_t0, :]

idx_S_t0 = np.random.choice(X_ns_t0.shape[0], N_S_t0, replace=False)
X_ns_t0 = X_ns_t0[idx_S_t0, :]
u1_t0 = u1_t0[idx_S_t0, :]
u2_t0 = u2_t0[idx_S_t0, :]
p_t0 = p_t0[idx_S_t0, :]

idx_darcy = np.random.choice(X_train_darcy_bc.shape[0], N_u_darcy_bc, replace=False)
X_train_darcy_bc = X_train_darcy_bc[idx_darcy, :]
phi_bc = phi_bc[idx_darcy, :]

idx_ns = np.random.choice(X_train_ns_bc.shape[0], N_u_ns_bc, replace=False)
X_train_ns_bc = X_train_ns_bc[idx_ns, :]
u1_bc = u1_bc[idx_ns, :]
u2_bc = u2_bc[idx_ns, :]
p_bc = p_bc[idx_ns, :]

idx_interface = np.random.choice(X_train_interface.shape[0], N_u_interface, replace=False)
X_train_interface = X_train_interface[idx_interface, :]

X_darcy_full = np.hstack([
    X_D_grid.reshape(-1, 1),
    Y_D_grid.reshape(-1, 1),
    T_D_grid.reshape(-1, 1)
])

y_ns_flat = Y_S_grid.reshape(-1, 1)
mask_ns_non_interface = (y_ns_flat < 0).flatten()

X_ns_non_interface = np.hstack([
    X_S_grid.reshape(-1, 1),
    Y_S_grid.reshape(-1, 1),
    T_S_grid.reshape(-1, 1)
])[mask_ns_non_interface, :]

X_star = np.vstack([X_darcy_full, X_ns_non_interface])



In [ ]:
"""Model construction"""


model = PhysicsInformedNN(
    X_darcy_t0, X_ns_t0,
    phi_t0, u1_t0, u2_t0, p_t0,
    X_train_darcy_bc, X_train_ns_bc, X_train_interface,
    phi_bc, u1_bc, u2_bc, p_bc,
    K, nu, g, alpha, s, nf, nd, tau, d, trace,
    X_darcy_lhs_pool,
    X_ns_lhs_pool,
    use_rl=True
)


print(f"   InitialNSarchitecture: depth={model.arch_config['ns_depth']}, width={model.arch_config['ns_width']}")
print(f"   InitialDarcyarchitecture: depth={model.arch_config['darcy_depth']}, width={model.arch_config['darcy_width']}")
print(f"   Initialsampling: Darcybase={model.N_darcy_base}, NSbase={model.N_ns_base}")
print(f"   Initialrefined: Darcy={model.N_darcy_refined}, NS={model.N_ns_refined}")


In [ ]:
"""Model training and summary"""



training_results = model.train(stage1_epochs=20000, stage2_epochs=10000)

print("\n" + "=" * 60)
print("Training summary")
print("=" * 60)
print(f"Total training time: {training_results['total_time']:.1f}s ({training_results['total_time'] / 60:.1f}min)")
print(f"Stage 1 time: {training_results['stage1_time']:.1f}s")
print(f"Stage 2 time: {training_results['stage2_time']:.1f}s")
print(f"\nFinal configuration:")
print(f"  NSarchitecture: depth={model.arch_config['ns_depth']}, width={model.arch_config['ns_width']}")
print(f"  Darcyarchitecture: depth={model.arch_config['darcy_depth']}, width={model.arch_config['darcy_width']}")
print(f"  Darcysampling: base{model.N_darcy_base} + refined{model.N_darcy_refined} = {model.N_darcy_base + model.N_darcy_refined}")
print(f"  NSsampling: base{model.N_ns_base} + refined{model.N_ns_refined} = {model.N_ns_base + model.N_ns_refined}")


In [ ]:
"""Prediction and timing summary"""



sync_cuda_if_available()
prediction_start_time = time.time()
phi_pred, u1_pred, u2_pred, p_pred = model.predict(X_star)
U_pred = np.hstack([phi_pred, u1_pred, u2_pred, p_pred])
sync_cuda_if_available()
prediction_end_time = time.time()
prediction_duration = prediction_end_time - prediction_start_time

print(f"Prediction completed! elapsed time: {prediction_duration:.2f}s")

print("\n" + "=" * 80)
print("                      Time statistics summary")
print("=" * 80)
print(f"Stage 1 training time:     {training_results['stage1_time']:.1f}s ({training_results['stage1_time'] / 60:.1f}min)")
print(f"Stage 2 training time:     {training_results['stage2_time']:.1f}s ({training_results['stage2_time'] / 60:.1f}min)")
print(f"Total training time:       {training_results['total_time']:.1f}s ({training_results['total_time'] / 60:.1f}min)")
print(f"Prediction time:         {prediction_duration:.2f}s")
print(
    f"Total computation time:       {training_results['total_time'] + prediction_duration:.1f}s ({(training_results['total_time'] + prediction_duration) / 60:.1f}min)")
print("=" * 80)

model.finalize_computational_cost_report(
    prediction_duration=prediction_duration,
    prediction_points=X_star.shape[0],
    prefix=COST_REPORT_PREFIX
)


In [ ]:
"""Multi-time visualization and error evaluation"""


t_targets = [0.2, 0.5, 0.8, 1.0]
t_values = T.flatten()

for t_target in t_targets:
    print(f"\nProcessing time point t = {t_target}")

    t_index = np.where(np.isclose(t_values, t_target, atol=1e-12))[0][0]

    Phi_t = Phi[:, :, t_index]
    U1_t = U1[:, :, t_index]
    U2_t = U2[:, :, t_index]
    P_t = P[:, :, t_index]

    mask_darcy_t = (X_star[:, 1] >= 0) & (X_star[:, 1] <= 0.75) & np.isclose(X_star[:, 2], t_target, atol=1e-12)
    mask_ns_t = (X_star[:, 1] >= -0.25) & (X_star[:, 1] <= 0) & np.isclose(X_star[:, 2], t_target, atol=1e-12)

    phi_pred_darcy = U_pred[mask_darcy_t, 0]
    u1_pred_ns = U_pred[mask_ns_t, 1]
    u2_pred_ns = U_pred[mask_ns_t, 2]
    p_pred_ns = U_pred[mask_ns_t, 3]

    error_phi_t = np.linalg.norm(Phi_t.flatten() - phi_pred_darcy) / np.linalg.norm(Phi_t.flatten())
    error_u1_t = np.linalg.norm(U1_t.flatten() - u1_pred_ns) / np.linalg.norm(U1_t.flatten())
    error_u2_t = np.linalg.norm(U2_t.flatten() - u2_pred_ns) / np.linalg.norm(U2_t.flatten())
    error_p_t = np.linalg.norm(P_t.flatten() - p_pred_ns) / np.linalg.norm(P_t.flatten())

    print(f't={t_target} relative error:')
    print(f'Error phi: {error_phi_t:.3e}')
    print(f'Error u1: {error_u1_t:.3e}')
    print(f'Error u2: {error_u2_t:.3e}')
    print(f'Error p: {error_p_t:.3e}')

    x_darcy = X_D_grid[0, :, t_index]
    y_darcy = Y_D_grid[:, 0, t_index]
    x_darcy_grid, y_darcy_grid = np.meshgrid(x_darcy, y_darcy)
    x_ns = X_S_grid[0, :, t_index]
    y_ns = Y_S_grid[:, 0, t_index]
    x_ns_grid, y_ns_grid = np.meshgrid(x_ns, y_ns)

    X_darcy_t = X_star[mask_darcy_t, :2]
    X_ns_t = X_star[mask_ns_t, :2]

    phi_pred_grid = griddata(X_darcy_t, phi_pred_darcy, (x_darcy_grid, y_darcy_grid), method='cubic')
    u1_pred_grid = griddata(X_ns_t, u1_pred_ns, (x_ns_grid, y_ns_grid), method='linear')
    u2_pred_grid = griddata(X_ns_t, u2_pred_ns, (x_ns_grid, y_ns_grid), method='linear')
    p_pred_grid = griddata(X_ns_t, p_pred_ns, (x_ns_grid, y_ns_grid), method='linear')

    phi_pred_grid = np.nan_to_num(phi_pred_grid, nan=0.0)
    u1_pred_grid = np.nan_to_num(u1_pred_grid, nan=0.0)
    u2_pred_grid = np.nan_to_num(u2_pred_grid, nan=0.0)
    p_pred_grid = np.nan_to_num(p_pred_grid, nan=0.0)

    phi_error = np.abs(Phi_t - phi_pred_grid)
    u1_error = np.abs(U1_t - u1_pred_grid)
    u2_error = np.abs(U2_t - u2_pred_grid)
    p_error = np.abs(P_t - p_pred_grid)

    field_names = ['phi', 'u1', 'u2', 'p']
    true_fields = [Phi_t, U1_t, U2_t, P_t]
    pred_fields = [phi_pred_grid, u1_pred_grid, u2_pred_grid, p_pred_grid]
    error_fields = [phi_error, u1_error, u2_error, p_error]

    grids_t = {
        'phi': (x_darcy_grid, y_darcy_grid),
        'u1': (x_ns_grid, y_ns_grid),
        'u2': (x_ns_grid, y_ns_grid),
        'p': (x_ns_grid, y_ns_grid)
    }

    for i, field in enumerate(field_names):
        plt.figure(figsize=(18, 5))

        if field == 'phi':
            X_grid, Y_grid = grids_t['phi']
        else:
            X_grid, Y_grid = grids_t['u1']

        plt.subplot(1, 3, 1)
        plt.pcolormesh(X_grid, Y_grid, true_fields[i], cmap='Spectral', shading='auto')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.title(f'Exact {field} at t={t_target}')
        plt.colorbar()

        plt.subplot(1, 3, 2)
        plt.pcolormesh(X_grid, Y_grid, pred_fields[i], cmap='Spectral', shading='auto')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.title(f'Predicted {field} at t={t_target}')
        plt.colorbar()

        plt.subplot(1, 3, 3)
        plt.pcolormesh(X_grid, Y_grid, error_fields[i], cmap='viridis', shading='auto')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.title(f'Absolute Error in {field} at t={t_target}')
        plt.colorbar()

        plt.tight_layout()
        filename = f'ns_darcy_{field}_t{t_target:.2f}_RL_dual_network_fixed.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        filename_eps = f'ns_darcy_{field}_t{t_target:.2f}_RL_dual_network_fixed.eps'
        plt.savefig(filename_eps, format='eps', bbox_inches='tight')
        plt.close()


In [ ]:
"""Loss curve plotting"""



plt.figure(figsize=(10, 6))
plt.semilogy(model.iteration_history, model.total_loss_history,
             'b-', linewidth=2.5, label='Total Loss', alpha=0.8)
plt.xlabel('Iteration', fontsize=13, fontweight='bold')
plt.ylabel('Total Loss (log scale)', fontsize=13, fontweight='bold')
plt.title('Total Loss Curve (Sum of All Seven Loss Terms)',
          fontsize=15, fontweight='bold', pad=15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('total_loss_curve_ns_darcy_fixed.png', dpi=300, bbox_inches='tight')
plt.savefig('total_loss_curve_ns_darcy_fixed.eps', format='eps', bbox_inches='tight')
plt.close()
print(" total losscurvesaved: total_loss_curve_ns_darcy_fixed.png / .eps")

plt.figure(figsize=(10, 6))
plt.semilogy(model.iteration_history, model.ns_loss_history,
             'g-', linewidth=2.5, label='NS Network Loss', alpha=0.8)
plt.xlabel('Iteration', fontsize=13, fontweight='bold')
plt.ylabel('NS Network Loss (log scale)', fontsize=13, fontweight='bold')
plt.title('NS Network Loss (NS PDE + NS BC + NS IC + Interface)',
          fontsize=15, fontweight='bold', pad=15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('ns_loss_curve_fixed.png', dpi=300, bbox_inches='tight')
plt.savefig('ns_loss_curve_fixed.eps', format='eps', bbox_inches='tight')
plt.close()


plt.figure(figsize=(10, 6))
plt.semilogy(model.iteration_history, model.darcy_loss_history,
             'r-', linewidth=2.5, label='Darcy Network Loss', alpha=0.8)
plt.xlabel('Iteration', fontsize=13, fontweight='bold')
plt.ylabel('Darcy Network Loss (log scale)', fontsize=13, fontweight='bold')
plt.title('Darcy Network Loss (Darcy PDE + Darcy BC + Darcy IC + Interface)',
          fontsize=15, fontweight='bold', pad=15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('darcy_loss_curve_fixed.png', dpi=300, bbox_inches='tight')
plt.savefig('darcy_loss_curve_fixed.eps', format='eps', bbox_inches='tight')
plt.close()
